# FloodVision GB10 — Notebook engine

This notebook is executed by `server.py` for uploaded images. Its image analysis, scoring, and report logic come from `FloodVision_GB10_English_Simple.ipynb`. Run it in JupyterLab with a `demo_config.json` file to test directly.

In [ ]:
import base64
import io
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from PIL import Image, ImageOps
try:
    from openai import OpenAI
except ImportError:
    from types import SimpleNamespace
    from urllib.request import Request, urlopen

    class OpenAI:
        def __init__(self, base_url, api_key, timeout=120):
            self.base_url = base_url.rstrip("/")
            self.api_key = api_key
            self.timeout = timeout
            self.chat = SimpleNamespace(completions=SimpleNamespace(create=self.create))

        def create(self, **payload):
            request = Request(
                f"{self.base_url}/chat/completions",
                data=json.dumps(payload).encode("utf-8"),
                headers={"Authorization": f"Bearer {self.api_key}", "Content-Type": "application/json"},
                method="POST",
            )
            with urlopen(request, timeout=self.timeout) as response:
                data = json.load(response)
            return SimpleNamespace(choices=[SimpleNamespace(message=SimpleNamespace(
                content=data["choices"][0]["message"]["content"]
            ))])

IP_ADDRESS = os.getenv("GB10_BASE_URL", "http://10.246.100.230/v1")
MODEL = os.getenv("GB10_MODEL", "qwen3-vl-8b-instruct")
client = OpenAI(base_url=IP_ADDRESS, api_key=os.getenv("GB10_API_KEY", "usaadatathon"), timeout=120)


In [ ]:
CONFIG_PATH = "demo_config.json"

In [ ]:
if Path(CONFIG_PATH).is_file():
    config = json.loads(Path(CONFIG_PATH).read_text(encoding="utf-8"))
else:
    config = {}

MODE = config.get("mode", "single")
IMAGE_PATHS = config.get("image_paths", [])
LOCATION = config.get("location", "Not specified")
VERIFIED_DRY_ROUTE = config.get("verified_dry_route", False)
DRY_ROUTE_DISTANCE_M = config.get("dry_route_distance_m")
RESULT_PATH = config.get("result_file")


In [ ]:
def image_to_url(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Image not found: {path}")
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image).convert("RGB")
        image.thumbnail((1280, 1280))
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG", quality=85)
    return "data:image/jpeg;base64," + base64.b64encode(buffer.getvalue()).decode("utf-8")


In [ ]:
PROMPT = """
You are analyzing visible flood conditions in photographs. Respond in English
Use only observable evidence. Do not guess water depth, current speed, distances, injury probability, or route safety.
Do not confuse ordinary rivers, lakes, or wet pavement with flooding.
If two images are provided, the first is BEFORE and the second is AFTER. Evaluate the AFTER image.
Use "yes", "no", or "uncertain" for each yes/no field. Flood extent must be "none", "localized", "widespread", or "uncertain".
A visible dry-looking exit does not prove the route is safe.
Identify potential places of refuge using only visible evidence. Compare elevated ground,
upper floors, rooftops, or visibly dry areas separated from floodwater. Return at most
three candidates, most suitable first, only when supported by the image. A dry-looking
street alone is insufficient evidence. Do not certify any place or route as safe or infer
accessibility, structural stability, water depth, current, or future conditions. Do not
recommend crossing floodwater, climbing onto roofs, or approaching electrical infrastructure.
For each candidate describe its visible evidence, visible hazards, and unknowns.
Locate each candidate with a tight bounding box [x_min, y_min, x_max, y_max], normalized
from 0 to 1000 relative to the full image (top-left origin). In comparison mode, locate
candidates in the AFTER image only. If none is supported, return an empty candidates
array and explain that no potential refuge can be reliably identified from this image.
All refuge descriptions and explanations must be in English.
Return ONLY a JSON object using this structure:
{
  "flooding_visible": "yes|no|uncertain",
  "flood_extent": "none|localized|widespread|uncertain",
  "road_or_path_flooded": "yes|no|uncertain",
  "moving_water_cues": "yes|no|uncertain",
  "visible_debris": "yes|no|uncertain",
  "people_exposed": "yes|no|uncertain",
  "vehicles_exposed": "yes|no|uncertain",
  "safe_dry_exit_visible": "yes|no|uncertain",
  "observations": ["2 to 4 visible facts"],
  "uncertainties": ["2 to 4 things the image cannot verify"],
  "summary": "Two-sentence summary without invented measurements",
  "potential_refuges": [{"label": "Descriptive place name", "bbox": [100, 100, 300, 300], "evidence": "Visible evidence supporting this candidate", "limitations": "Visible hazards and what cannot be verified"}],
  "refuge_assessment": "Explain the first candidate's relative suitability or why no potential refuge can be reliably identified",
  "change_from_before": "Visible changes if there are two photos; otherwise empty string"
}
""".strip()

FIELDS = [
    "flooding_visible", "road_or_path_flooded", "moving_water_cues",
    "visible_debris", "people_exposed", "vehicles_exposed", "safe_dry_exit_visible"
]


def parse_response(text):
    text = text.strip()
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        if start < 0:
            raise ValueError(f"No JSON found in the model response: {text[:200]}")
        data, _ = json.JSONDecoder().raw_decode(text[start:])

    if not isinstance(data, dict):
        raise ValueError("Expected a JSON object")
    for field in FIELDS:
        value = str(data.get(field, "uncertain")).lower().strip()
        data[field] = value if value in {"yes", "no", "uncertain"} else "uncertain"
    extent = str(data.get("flood_extent", "uncertain")).lower().strip()
    data["flood_extent"] = extent if extent in {"none", "localized", "widespread", "uncertain"} else "uncertain"
    for field in ("observations", "uncertainties"):
        value = data.get(field, [])
        data[field] = [str(item) for item in value[:6] if isinstance(item, str)] if isinstance(value, list) else []
    for field in ("summary", "change_from_before"):
        data[field] = str(data.get(field, ""))
    candidates = data.get("potential_refuges", [])
    valid = []
    for candidate in candidates[:3] if isinstance(candidates, list) else []:
        if not isinstance(candidate, dict):
            continue
        box = candidate.get("bbox")
        if (not isinstance(box, list) or len(box) != 4
            or any(isinstance(v, bool) or not isinstance(v, (int, float)) or not 0 <= v <= 1000 for v in box)
            or box[0] >= box[2] or box[1] >= box[3]):
            continue
        if not all(isinstance(candidate.get(k), str) and candidate[k].strip()
                   for k in ("label", "evidence", "limitations")):
            continue
        valid.append({k: candidate[k] for k in ("label", "bbox", "evidence", "limitations")})
    data["potential_refuges"] = valid
    assessment = data.get("refuge_assessment")
    data["refuge_assessment"] = (
        assessment.strip() if valid and isinstance(assessment, str) and assessment.strip()
        else "Potential refuge candidates require independent verification." if valid
        else "No potential refuge can be reliably identified from this image."
    )
    if data["flooding_visible"] == "no" and (
        data["road_or_path_flooded"] == "yes" or data["moving_water_cues"] == "yes"
    ):
        data["flooding_visible"] = "uncertain"
        data["uncertainties"].append("The observations conflict and need review.")
    return data


def analyze_images(paths):
    content = [{"type": "text", "text": PROMPT}]
    for i, path in enumerate(paths):
        if len(paths) == 2:
            content.append({"type": "text", "text": "BEFORE" if i == 0 else "AFTER"})
        content.append({"type": "image_url", "image_url": {"url": image_to_url(path)}})

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Describe only visible evidence. Return valid JSON. Never certify safety."},
            {"role": "user", "content": content},
        ],
        temperature=0.001,
        max_tokens=2200,
    )
    return parse_response(response.choices[0].message.content or "")

In [ ]:
HAZARD_WEIGHTS = [
    ("flood_extent", "widespread", 8, "Widespread water"),
    ("road_or_path_flooded", "yes", 15, "Flooded road or path"),
    ("moving_water_cues", "yes", 18, "Visible signs of moving water"),
    ("visible_debris", "yes", 8, "Debris or obstacles"),
    ("people_exposed", "yes", 8, "People in the affected area"),
    ("vehicles_exposed", "yes", 7, "Vehicles in the affected area"),
    ("safe_dry_exit_visible", "no", 12, "No dry-looking exit visible"),
]


def hazard_indicator(data):
    if data["flooding_visible"] != "yes":
        return {"score": None, "level": "Not assessable", "reasons": ["Flooding is not confirmed in the image"]}
    score = 50
    reasons = ["Visible evidence of flooding"]
    for field, expected, points, reason in HAZARD_WEIGHTS:
        if data.get(field) == expected:
            score += points
            reasons.append(f"{reason} (+{points})")
    score = min(score, 100)
    return {"score": score, "level": "Critical" if score >= 80 else "High", "reasons": reasons}


def dry_route_time(distance_m, verified):
    if not verified or distance_m is None:
        return None
    if isinstance(distance_m, bool) or not isinstance(distance_m, (int, float)) or not 0 < distance_m <= 100_000:
        raise ValueError("Enter a valid dry-route distance in meters")
    return {
        "distance_m": distance_m,
        "min_minutes": round(distance_m / 1.2 / 60, 1),
        "max_minutes": round(distance_m / 0.6 / 60, 1),
    }


def build_report(data, paths):
    return {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "location": LOCATION,
        "images": [str(path) for path in paths],
        "model": MODEL,
        "analysis": data,
        "visual_hazard": hazard_indicator(data),
        "dry_route_time": dry_route_time(DRY_ROUTE_DISTANCE_M, VERIFIED_DRY_ROUTE),
    }

In [ ]:
def report_markdown(report):
    analysis = report["analysis"]
    hazard = report["visual_hazard"]
    travel = report["dry_route_time"]
    score = f"{hazard['score']}/100 points ({hazard['level']})" if hazard["score"] is not None else "Not assessable"
    time_text = (
        f"{travel['min_minutes']}–{travel['max_minutes']} minutes for {travel['distance_m']} m "
        "on an independently verified dry route (illustrative only)"
        if travel else "Not available: dry-route distance and independent verification required"
    )
    lines = [
        "# FloodVision GB10 — Image Report", "",
        f"**Location:** {report['location']}",
        f"**Images:** {', '.join(report['images'])}",
        f"**Summary:** {analysis['summary']}", "",
        f"**Flood visible:** {analysis['flooding_visible']}",
        f"**Flood extent:** {analysis['flood_extent']}",
        f"**Flooded road/path:** {analysis['road_or_path_flooded']}",
        f"**Signs of moving water:** {analysis['moving_water_cues']}",
        f"**Visual hazard indicator:** {score}",
        f"**Dry-route walking time:** {time_text}", "",
        "## Visual evidence",
        *[f"- {item}" for item in analysis["observations"]], "",
        "## Hazard indicators",
        *[f"- {item}" for item in hazard["reasons"]], "",
        "## Uncertainties",
        *[f"- {item}" for item in analysis["uncertainties"]],
    ]
    lines.extend(["", "## Potential places of refuge", analysis["refuge_assessment"]])
    for index, candidate in enumerate(analysis["potential_refuges"], 1):
        lines.extend([
            f"### {index}. {candidate['label']}",
            f"Visible evidence: {candidate['evidence']}",
            f"Hazards and unknowns: {candidate['limitations']}",
            f"Approximate image box (0–1000): {candidate['bbox']}",
        ])
    lines.append("Candidates and image markers are model estimates, not verified safe destinations or routes.")
    if analysis["change_from_before"]:
        lines.extend(["", "## Before and after", analysis["change_from_before"]])
    lines.extend([
        "", "**Safety:** This experimental visual hazard indicator is not an injury probability or a validated safety rating. "
        "A photo cannot verify the depth, current, or whether a route is safe. Never cross floodwater."
    ])
    return "\n".join(lines)


In [ ]:
reports = []
if IMAGE_PATHS:
    if MODE == "before_after":
        if len(IMAGE_PATHS) != 2:
            raise ValueError("Before/after analysis requires exactly two images")
        reports.append(build_report(analyze_images(IMAGE_PATHS), IMAGE_PATHS))
    elif MODE == "single":
        for path in IMAGE_PATHS:
            reports.append(build_report(analyze_images([path]), [path]))
    else:
        raise ValueError("Invalid analysis mode")

for report in reports:
    report["images"] = [Path(name).name for name in report["images"]]
    report["visual_hazard"]["score_type"] = "illustrative_points_not_probability"

if RESULT_PATH:
    Path(RESULT_PATH).write_text(json.dumps({
        "reports": reports,
        "markdown": [report_markdown(report) for report in reports],
    }, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Analyzed {len(reports)} report(s)")
